# **Imports & Setup**

In [ ]:
# @title Imports
# hide noisy output from pip install
%%capture

!pip install transformers datasets optuna tensorboard

import pandas as pd
import os
from datasets import Dataset
from transformers import AutoTokenizer, TrainingArguments, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, Trainer, AutoConfig, EarlyStoppingCallback, set_seed
import torch
import random
import numpy as np
import matplotlib.pyplot as plt

import optuna
import json
from google.colab import drive

## Setup

Due to the resource intensive model training, this notebook is intended to be run in Google Colab using a T4 GPU runtime.

To import the files that are required for the code below (e.g. training data files), copy them into your Google Drive and connect it to this notebook.

In [ ]:
# Input data files are stored on Google Drive to make them usable within Colab
# Mount Google Drive to access data
drive.mount("/content/drive")

DRIVE_BASE_PATH = "Please enter path to data folder" #@param

In [ ]:
# Define train-test split ratio
TRAIN_RATIO = 0.8 # @param
# Ensures reproducibility of data generation
RANDOM_SEED = 42   # @param

# Set seeds for full reproducibility
def set_seed(seed=RANDOM_SEED):
  """ Set all random seeds for reproducibility."""
  random.seed(RANDOM_SEED)
  np.random.seed(RANDOM_SEED)
  torch.manual_seed(RANDOM_SEED)
  torch.cuda.manual_seed_all(RANDOM_SEED)
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False

set_seed()


# Load the model and tokenizer
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# **Data Loading and Preprocessing**

In [ ]:
# Load training dataset
df = pd.read_csv(f"{DRIVE_BASE_PATH}/data/train_dataset.csv")

In [ ]:
# Modify dataframe to include the column input and output specifically for T5 model
df["input_text"] = "Context: " + df["paragraph"] + " Question: " + df["question"]
df["output_text"] = df["answer"]

In [ ]:
# Tokenize function
def tokenize_data(example):
  inputs = tokenizer(
      example["input_text"], max_length=512, truncation=True, padding="max_length"
  )
  labels = tokenizer(
      example["output_text"], max_length=128, truncation=True, padding="max_length"
  )
  inputs["labels"] = labels["input_ids"]
  return inputs

In [ ]:
# Convert dataframe to Hugging Face Dataset
all_dataset = Dataset.from_pandas(df[["input_text", "output_text"]])
# Tokenize dataset
tokenized_dataset = all_dataset.map(tokenize_data, batched=True, remove_columns=all_dataset.column_names)

In [ ]:
# Split tokenized dataset into train and test
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=RANDOM_SEED)

# Access train and test subsets
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")

# **Hyperparameter Optimization with Optuna**

In [ ]:
# Define Optuna objective function
def objective(trial):
    """Objective function for Optuna hyperparameter tuning with manual pruning."""

    # Hyperparameters for optimization
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 3e-4)
    batch_size = trial.suggest_categorical("batch_size", [8, 16])
    num_train_epochs = trial.suggest_int("num_train_epochs", 3, 5)
    weight_decay = trial.suggest_loguniform("weight_decay", 0.01, 0.02)

    # Load the model from scratch for each trial
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

    # Fine-tuning configuration
    training_args = Seq2SeqTrainingArguments(
        output_dir=f"./optuna_tuned_model_trial_{trial.number}",
        evaluation_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        save_strategy="epoch",
        weight_decay=weight_decay,
        save_total_limit=2,
        logging_dir="./logs",
        logging_steps=1,
        report_to="none",
        predict_with_generate=True,
        lr_scheduler_type="cosine",  # Using a cosine learning rate decay to slowly reduce the learning rate over time
        fp16=True,    #Enables 16-bit floating point (mixed precision) training
    )

    # Initialize Trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        tokenizer=tokenizer
        )

    # Train and Evaluate
    trainer.train()
    eval_results = trainer.evaluate()

    # Report loss to Optuna
    trial.report(eval_results["eval_loss"], step=num_train_epochs)  # the objective function returns eval_results["eval_loss"], optuna will try to minimize eval loss as much as possible

    # Apply manual pruning
    if trial.should_prune():
        raise optuna.exceptions.TrialPruned()

    # Free GPU memory after each trial
    del model
    torch.cuda.empty_cache()

    return eval_results["eval_loss"]

In [ ]:
# @title Run Optuna study and find best hyperparameters

# Create Optuna study (no parallel execution, not enough memory)
study = optuna.create_study(
    direction="minimize",  # The goal is to find the lowest possible value of the objective function (eval loss)
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=1),  # Prunes unpromising trials
    sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)  # Fixes randomness in sampling for reproducibility
)

# Run optimization
study.optimize(objective, n_trials=10)

print("Total trials completed:", len(study.trials))
print("Pruned trials:", sum([t.state == optuna.trial.TrialState.PRUNED for t in study.trials]))

print("Best Hyperparameters:", study.best_params)

In [ ]:
# Path in Google Drive to store the best hyperparameters
save_path = f"{DRIVE_BASE_PATH}/t5_small/best_hyperparams_t5small.json"

# Save best hyperparameters to Google Drive
with open(save_path, "w") as f:
    json.dump(study.best_params, f)

print(f"Best hyperparameters saved to {save_path} in Google Drive.")

# **Fine-Tune Model**

In [ ]:
# Optional starting point-- Define path where file was saved
load_path = f"{DRIVE_BASE_PATH}/t5_small/best_hyperparams_t5small.json"

# Load best hyperparameters
with open(load_path, "r") as f:
    best_params = json.load(f)

print("Best hyperparameters reloaded:", best_params)

In [ ]:
# Reload the pretrained model for fine-tuning
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to("cuda")

# Train final model using the best hyperparameters
final_training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-small-qa_dk90",
    evaluation_strategy="epoch",
    seed=RANDOM_SEED,
    learning_rate=best_params["learning_rate"],
    per_device_train_batch_size=best_params["batch_size"],
    per_device_eval_batch_size=best_params["batch_size"],
    num_train_epochs=best_params["num_train_epochs"],
    weight_decay=best_params["weight_decay"],
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=1,
    report_to="tensorboard",
    lr_scheduler_type="cosine",
    load_best_model_at_end=True,
    predict_with_generate=True,
)

final_trainer = Seq2SeqTrainer(
    model=model,
    args=final_training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# Set up Tensorboard to monitor eval loss, learning rate and training loss
%load_ext tensorboard
%tensorboard --logdir=./logs

In [ ]:
# Train the final model using the best hyperparameters
final_trainer.train()

In [ ]:
# Run evaluation and print eval loss at selected checkpoint
eval_results = final_trainer.evaluate()
print(f"Eval loss at {final_trainer.state.best_model_checkpoint}: {eval_results['eval_loss']}")

In [ ]:
# Define where to save the model
save_directory = f"{DRIVE_BASE_PATH}/t5_small/small_qa_finetuned"

# Save the fine-tuned model and tokenizer
final_trainer.model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"Fine-tuned model saved to {save_directory}")

# **Load Saved Fine-Tuned Model**

In [ ]:
# Load the fine-tuned model and tokenizer
model_path = f"{DRIVE_BASE_PATH}/t5_small/t5-small_qa_finetuned"
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("Fine-tuned model loaded successfully!")

In [ ]:
# Move model to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print(f"Fine-tuned model loaded successfully on {device}!")

# **Prompt Fine-Tuned Model to Generate Three Answers per Question**

In [ ]:
# Function to generate three answers per question for the test data
def generate_answers_from_testset(test_dataset, model, tokenizer):
    results = []

    # Iterate over each sample in the test dataset
    for _, row in test_dataset.iterrows():
        # Construct input text from dataset columns
        test_input = f"Context: {row['paragraph']} Question: {row['question']}"

        # Tokenize the input
        inputs = tokenizer(test_input, return_tensors="pt", truncation=True).to("cuda")

        # Generate three answers
        outputs = model.generate(inputs["input_ids"],
                                 max_length=300,
                                 num_return_sequences=3,  # Generate 3 outputs
                                 do_sample=True,
                                 top_k=10,
                                 temperature=0.7)

        # Decode and clean generated responses
        generated_answers = [
            tokenizer.decode(output, skip_special_tokens=True).replace(test_input, "").strip()
            for output in outputs
        ]

        # Append results to list
        results.append({
            "Context": row['paragraph'],
            "Question": row['question'],
            "Ground Truth Answer": row['answer'],  # Include the correct answer
            "Answer 1": generated_answers[0],
            "Answer 2": generated_answers[1],
            "Answer 3": generated_answers[2]
        })

    # Convert results into a DataFrame
    results_df = pd.DataFrame(results)

    return results_df

In [ ]:
test_dataset_path = f"{DRIVE_BASE_PATH}/data/test_dataset.csv"
generated_answers_path = f"{DRIVE_BASE_PATH}/t5_small/modelanswers_t5small.csv"

test_dataset = pd.read_csv(test_dataset_path)  # Load dataset
generated_responses_df = generate_answers_from_testset(test_dataset, model, tokenizer)
generated_responses_df.to_csv(generated_answers_path)

# **Model to Verify Each Generated Answer**

In [ ]:
# Function to verify generated answers by feeding them back into the model
def validate_generated_answers(csv_file, model, tokenizer):
    # Load the dataset containing Context, Question, and generated answers
    df = pd.read_csv(csv_file)

    # Iterate over each row and validate answers
    for i in range(1, 4):  # Answer 1, Answer 2, Answer 3
        validation_results = []  # Store model validation outputs

        for _, row in df.iterrows():
            question = row["Question"]
            context = row["Context"]
            answer_text = row[f"Answer {i}"]

            # Construct the validation input in the correct format
            validation_input = f" Context: {context} Question: {answer_text}, is that correct?"

            # Tokenize and generate validation response
            inputs = tokenizer(validation_input, return_tensors="pt", truncation=True).to("cuda")
            validation_output = model.generate(inputs["input_ids"], max_length=5, num_return_sequences=1, do_sample=False, top_k=0, temperature=0)

            # Decode and store validation answer
            validation_answer = tokenizer.decode(validation_output[0], skip_special_tokens=True).replace(validation_input, "").strip()
            validation_results.append(validation_answer)

        # Append the validation results as a new column in the original dataframe
        df[f"Model Validation Answer (Answer {i})"] = validation_results

    return df=

In [ ]:
validated_answers_path = f"{DRIVE_BASE_PATH}/t5_small/modelanswersvalidated_t5small.csv"

validated_responses_df = validate_generated_answers(generated_answers_path, model, tokenizer)
validated_responses_df.to_csv(validated_answers_path)